# TIGER SemanticID on Amazon Datasets — Experiment Plan

Goal: Implement Semantic IDs via RQ-VAE and a compact seq2seq Transformer for generative retrieval on Amazon 5-core datasets; produce metrics and visualizations validating paper claims.

Datasets: Amazon Product Reviews (Beauty, Video_Games). Switch between datasets using the `dataset_name` config parameter.

Key steps: Download & preprocess; Sentence-T5 embeddings; RQ-VAE (3 levels, K=256) to 3-tuple codes + collision code c4; visualizations (c1↔category, hierarchy); seq2seq generative retrieval; metrics Recall@5/10, NDCG@5/10 and invalid-ID rate; ablations (Random/LSH); mini cold-start probe.

Artifacts: save to /content/artifacts. Keep configs modest for Colab; add knobs for smoke tests.

In [1]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Setup working directory in Google Drive
import os
assert os.path.exists('/content/drive')
WORK_DIR = '/content/drive/MyDrive/colab/tiger_semantic_id'
%mkdir -p $WORK_DIR

In [3]:
# Create RQ-VAE building folder
import shutil
import os

RQVAE_DIR = WORK_DIR + '/rq_vae_building'

# Remove existing folder if it exists
if os.path.exists(RQVAE_DIR):
    print(f"Removing existing {RQVAE_DIR} folder...")
    shutil.rmtree(RQVAE_DIR)

# Create fresh folder
os.makedirs(RQVAE_DIR)
print(f"✅ Created fresh {RQVAE_DIR} folder")

Removing existing /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building folder...
✅ Created fresh /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building folder


In [4]:
# Specify data preparation artifacts location
DATA_PREP_DIR = WORK_DIR + '/data_preparation/artifacts'

print("=" * 60)
print("DATA DEPENDENCY CONFIGURATION")
print("=" * 60)
print(f"\nData source: {DATA_PREP_DIR}")
print(f"Exists: {os.path.exists(DATA_PREP_DIR)}")

if not os.path.exists(DATA_PREP_DIR):
    print("\n⚠️  Data preparation artifacts not found!")
    print("   Please run TIGER_SemanticID_data_preparation.ipynb first")
else:
    print("\n✅ Data preparation artifacts found")
    print("\nAvailable files:")
    for f in sorted(os.listdir(DATA_PREP_DIR)):
        filepath = os.path.join(DATA_PREP_DIR, f)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"  - {f} ({size_mb:.2f} MB)")

print("=" * 60)

DATA DEPENDENCY CONFIGURATION

Data source: /content/drive/MyDrive/colab/tiger_semantic_id/data_preparation/artifacts
Exists: True

✅ Data preparation artifacts found

Available files:
  - config.json (0.00 MB)
  - item2id.json (0.23 MB)
  - item_embeddings.pt (35.45 MB)
  - item_texts.json (1.58 MB)
  - items.pkl (2.01 MB)
  - test_df.pkl (1.15 MB)
  - train_df.pkl (7.11 MB)
  - user2id.json (0.52 MB)
  - val_df.pkl (1.15 MB)


In [5]:
# Load prepared data from data preparation notebook
import pickle
import torch
import json
import pandas as pd
import sys

print("=" * 60)
print("LOADING PREPARED DATA")
print("=" * 60)

# 1. Load config
print("\n[1] Loading configuration...")
with open(f"{DATA_PREP_DIR}/config.json", "r") as f:
    config_dict = json.load(f)

from dataclasses import dataclass
@dataclass
class Config:
    dataset_name: str = 'Beauty'
    min_user_interactions: int = 5
    max_hist_len: int = 20
    embed_model_name: str = 'sentence-t5-base'
    rqvae_latent_dim: int = 32
    rqvae_levels: int = 3
    rqvae_codebook_size: int = 256
    rqvae_beta: float = 0.0025
    rqvae_alpha: float = 0.01
    rqvae_epochs: int = 1000
    rqvae_batch_size: int = 1024
    rqvae_lr: float = 1e-3
    seq2seq_d_model: int = 128
    seq2seq_ff: int = 1024
    seq2seq_heads: int = 8
    seq2seq_layers_enc: int = 4
    seq2seq_layers_dec: int = 4
    seq2seq_dropout: float = 0.1
    seq2seq_batch_size: int = 256
    seq2seq_steps: int = 20000
    seq2seq_lr: float = 1e-2
    user_vocab_hash: int = 2000
    topk_list: tuple = (5, 10)

cfg = Config(**config_dict)
print(f"✅ Loaded config: dataset={cfg.dataset_name}")

# 2. Load DataFrames
print("\n[2] Loading DataFrames...")
train_df = pd.read_pickle(f"{DATA_PREP_DIR}/train_df.pkl")
val_df = pd.read_pickle(f"{DATA_PREP_DIR}/val_df.pkl")
test_df = pd.read_pickle(f"{DATA_PREP_DIR}/test_df.pkl")
items = pd.read_pickle(f"{DATA_PREP_DIR}/items.pkl")
print(f"✅ train_df: {train_df.shape}")
print(f"✅ val_df: {val_df.shape}")
print(f"✅ test_df: {test_df.shape}")
print(f"✅ items: {items.shape}")

# 3. Load embeddings
print("\n[3] Loading item embeddings...")
item_emb = torch.load(f"{DATA_PREP_DIR}/item_embeddings.pt")
print(f"✅ item_emb: {item_emb.shape}")
print(f"   Device: {item_emb.device}")

# 4. Load ID mappings
print("\n[4] Loading ID mappings...")
with open(f"{DATA_PREP_DIR}/user2id.json", "r") as f:
    user2id = json.load(f)
with open(f"{DATA_PREP_DIR}/item2id.json", "r") as f:
    item2id = json.load(f)
print(f"✅ user2id: {len(user2id)} users")
print(f"✅ item2id: {len(item2id)} items")

# 5. Setup Paths for artifacts
print("\n[5] Setting up paths...")
class Paths:
    data_dir = DATA_PREP_DIR  # Reuse data prep artifacts
    artifacts_dir = "./artifacts"  # Save new artifacts (RQ-VAE, etc.) locally

import os
os.makedirs(Paths.artifacts_dir, exist_ok=True)
print(f"✅ Artifacts will be saved to: {Paths.artifacts_dir}")

# 6. Setup src imports
print("\n[6] Setting up src imports...")
# Clone repo for src code
repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = '20250908_tiger_dev'

try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists(repo_dir):
        !rm -rf $repo_dir
    !git clone -q $repo_url
    %cd $repo_dir
    !git fetch --all
    !git checkout $branch_name

    src_path = os.path.abspath('tiger_semantic_id/src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    print(f"✅ Added src to path: {src_path}")
else:
    print("ℹ️  Running locally (not Colab)")

print("\n" + "=" * 60)
print("DATA LOADING COMPLETE")
print("=" * 60)
print("\n✅ Ready to start RQ-VAE training!")

LOADING PREPARED DATA

[1] Loading configuration...
✅ Loaded config: dataset=Beauty

[2] Loading DataFrames...
✅ train_df: (138622, 5)
✅ val_df: (22363, 5)
✅ test_df: (22363, 5)
✅ items: (12101, 7)

[3] Loading item embeddings...
✅ item_emb: torch.Size([12101, 768])
   Device: cpu

[4] Loading ID mappings...
✅ user2id: 22363 users
✅ item2id: 12101 items

[5] Setting up paths...
✅ Artifacts will be saved to: ./artifacts

[6] Setting up src imports...
/content/recsys_playground
Fetching origin
Branch '20250908_tiger_dev' set up to track remote branch '20250908_tiger_dev' from 'origin'.
Switched to a new branch '20250908_tiger_dev'
✅ Added src to path: /content/recsys_playground/tiger_semantic_id/src

DATA LOADING COMPLETE

✅ Ready to start RQ-VAE training!


In [6]:
# The improved RQ-VAE architecture is now integrated into the main RQVAE class in rqvae.py
# No need for separate ImprovedRQVAE class or train_rqvae patches
print("✓ Using improved RQ-VAE architecture from rqvae.py")

✓ Using improved RQ-VAE architecture from rqvae.py


In [ ]:
# RQ-VAE training with corrected hyperparameters (post-diagnostic fix)
import torch
from tiger_semantic_id.src.rqvae import RQVAE, RQVAEConfig, train_rqvae, encode_codes

# Setup device for GPU acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"RQ-VAE training will use device: {device}")

# Create RQ-VAE configuration with CORRECTED PARAMETERS
rqcfg = RQVAEConfig(input_dim=item_emb.shape[1], latent_dim=cfg.rqvae_latent_dim, levels=cfg.rqvae_levels, codebook_size=cfg.rqvae_codebook_size)
rqcfg.beta = cfg.rqvae_beta
rqcfg.alpha = cfg.rqvae_alpha

print("=== CREATING RQ-VAE MODEL (POST-DIAGNOSTIC FIX) ===")
model = RQVAE(rqcfg).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

# Move embeddings to same device as training will occur
if item_emb.device != torch.device(device):
    print(f"Moving embeddings from {item_emb.device} to {device}")
    item_emb = item_emb.to(device)
else:
    print(f"Embeddings already on {device}")

print("\n=== TESTING MODEL DIVERSITY (PRE-TRAINING) ===")
# Test diversity before training to ensure the improved architecture works
with torch.no_grad():
    # Test encoder diversity
    encoded = model.encoder(model.normalize(item_emb[:20]))
    dists = torch.cdist(encoded[:10], encoded[:10])
    print(f"Encoder diversity: mean pairwise distance = {dists.fill_diagonal_(float('inf')).mean():.6f}")

    # Test quantization diversity
    z = model.encoder(model.normalize(item_emb[:100]))
    # No LayerNorm - removed to prevent numerical explosion
    q, codes = model.codebook(z)
    unique_codes = len(torch.unique(codes, dim=0))
    print(f"Quantization diversity: {unique_codes} unique codes out of 100 items")
    print(f"Sample codes: {codes[:5].tolist()}")

# Run RQ-VAE training with CORRECTED PARAMETERS:
# 1. Adam optimizer (lr=1e-3) - stable, proven
# 2. Beta=0.25 - prevents encoder collapse
# 3. 50 epochs - sufficient for convergence
# 4. NO per-level residual normalization (removed - caused numerical explosion)
print(f"\n=== STARTING RQ-VAE TRAINING ON {device.upper()} ===")
print(f"Training parameters: epochs={cfg.rqvae_epochs}, batch_size={cfg.rqvae_batch_size}, lr={cfg.rqvae_lr}, alpha={cfg.rqvae_alpha}, beta={cfg.rqvae_beta}, optimizer=Adam")
print("KEY FIXES:")
print("  ✅ CRITICAL: Removed LayerNorm (amplified encoder variance ~0.005 into 400k+ distances)")
print("  ✅ Removed per-level residual normalization (same issue as LayerNorm)")
print("  ✅ Reverted to Adam optimizer with lr=1e-3 (Adagrad@0.4 caused encoder collapse)")
print("  ✅ Reduced beta to 0.25 (high beta encourages encoder to output constants)")
print("  ✅ Reduced epochs to 50 (long training in bad basin doesn't help)")

# Adjust batch size based on device (larger for GPU)
training_batch_size = cfg.rqvae_batch_size if device == "cuda" else min(cfg.rqvae_batch_size, 512)
if training_batch_size != cfg.rqvae_batch_size:
    print(f"Adjusted batch size for {device}: {cfg.rqvae_batch_size} -> {training_batch_size}")

model = train_rqvae(
    model,
    item_emb,
    epochs=cfg.rqvae_epochs,
    batch_size=training_batch_size,
    lr=cfg.rqvae_lr,
    device=device,
    optimizer="adam",           # CORRECTED: Back to Adam
    revive_every=10,            # CORRECTED: Less aggressive revival
    revive_threshold=5
)

# Save trained model
torch.save(model.state_dict(), f"{Paths.artifacts_dir}/rqvae.pt")
print(f"Saved trained model to {Paths.artifacts_dir}/rqvae.pt")

# Generate codes for the full dataset
print("\n=== GENERATING SEMANTIC CODES ===")
codes = encode_codes(model, item_emb, device=device)
final_unique = len(torch.unique(codes, dim=0))
print(f"Final codes shape: {codes.shape}")
print(f"Final unique codes: {final_unique} out of {len(codes)} items ({100*final_unique/len(codes):.1f}% diversity)")
print(f"Sample final codes: {codes[:5].tolist()}")

# Check if diversity was preserved through training
if final_unique > len(codes) * 0.8:  # More than 80% unique
    print("✅ Excellent code diversity preserved through training!")
elif final_unique > len(codes) * 0.5:  # More than 50% unique
    print("✅ Good code diversity maintained")
else:
    print("⚠️  Code diversity may need improvement")

RQ-VAE training will use device: cpu
=== CREATING RQ-VAE MODEL (POST-DIAGNOSTIC FIX) ===
Model created with 493088 parameters
Embeddings already on cpu

=== TESTING MODEL DIVERSITY (PRE-TRAINING) ===
Encoder diversity: mean pairwise distance = inf
Quantization diversity: 84 unique codes out of 100 items
Sample codes: [[69, 178, 80], [79, 52, 52], [94, 178, 80], [84, 158, 194], [229, 235, 80]]

=== STARTING RQ-VAE TRAINING ON CPU ===
Training parameters: epochs=1000, batch_size=1024, lr=0.001, alpha=0.01, beta=0.0025, optimizer=Adam
KEY FIXES:
  ✅ CRITICAL: Removed LayerNorm (amplified encoder variance ~0.005 into 400k+ distances)
  ✅ Removed per-level residual normalization (same issue as LayerNorm)
  ✅ Reverted to Adam optimizer with lr=1e-3 (Adagrad@0.4 caused encoder collapse)
  ✅ Reduced beta to 0.25 (high beta encourages encoder to output constants)
  ✅ Reduced epochs to 50 (long training in bad basin doesn't help)
Adjusted batch size for cpu: 1024 -> 512
Using legacy device strin

In [ ]:
# DIAGNOSTIC: Test encoder-decoder WITHOUT quantization (pure autoencoder)
print("=" * 60)
print("AUTOENCODER TEST (No Quantization)")
print("=" * 60)

import torch
import torch.nn.functional as F
from tiger_semantic_id.src.rqvae import RQVAE, RQVAEConfig

# Create a fresh model
device = "cuda" if torch.cuda.is_available() else "cpu"
test_cfg = RQVAEConfig(input_dim=item_emb.shape[1], latent_dim=cfg.rqvae_latent_dim, levels=3, codebook_size=256)
test_model = RQVAE(test_cfg).to(device)

# Fit normalizer
test_model.x_mean.copy_(item_emb.mean(dim=0, keepdim=True))
test_model.x_std.copy_(item_emb.std(dim=0, keepdim=True))

# Train WITHOUT quantization - just encoder -> decoder
optimizer = torch.optim.Adam(list(test_model.encoder.parameters()) + list(test_model.decoder.parameters()), lr=1e-3)

print(f"\nTraining pure autoencoder (latent_dim={cfg.rqvae_latent_dim}, no quantization)...\n")

test_model.train()
batch_size = 1024
N = item_emb.size(0)

for epoch in range(1, 21):
    perm = torch.randperm(N, device=device)
    total_loss = 0.0

    for i in range(0, N, batch_size):
        batch = item_emb[perm[i:i+batch_size]]

        # Normalize input
        x_norm = (batch - test_model.x_mean) / (test_model.x_std + 1e-8)

        # Encoder -> Decoder (NO quantization)
        z = test_model.encoder(x_norm)
        x_recon = test_model.decoder(z)

        # MSE loss
        loss = F.mse_loss(x_recon, x_norm)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(test_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * batch.size(0)

    avg_loss = total_loss / N

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/20: MSE = {avg_loss:.6f}")

print("\n" + "=" * 60)
print("INTERPRETATION:")
print("=" * 60)
if avg_loss < 0.5:
    print("✅ MSE decreased significantly - encoder/decoder CAN learn!")
    print("   Problem is likely in the quantization mechanism.")
elif avg_loss < 0.9:
    print("⚠️  MSE decreased but not great - may need bigger latent_dim")
else:
    print("❌ MSE stuck ~1.0 - encoder/decoder architecture issue")
    print("   Problem is NOT the quantization.")
print("=" * 60)


In [ ]:
# DIAGNOSTIC: Test WITH quantization but WITHOUT VQ losses (recon only)
print("=" * 60)
print("QUANTIZATION TEST (Reconstruction Loss ONLY)")
print("=" * 60)

import torch
import torch.nn.functional as F
from tiger_semantic_id.src.rqvae import RQVAE, RQVAEConfig

# Create a fresh model
device = "cuda" if torch.cuda.is_available() else "cpu"
test_cfg = RQVAEConfig(input_dim=item_emb.shape[1], latent_dim=cfg.rqvae_latent_dim, levels=3, codebook_size=256)
test_model = RQVAE(test_cfg).to(device)

# Fit normalizer
test_model.x_mean.copy_(item_emb.mean(dim=0, keepdim=True))
test_model.x_std.copy_(item_emb.std(dim=0, keepdim=True))

# K-means init codebooks
print("Initializing codebooks with k-means...")
with torch.no_grad():
    ridx = torch.randperm(item_emb.size(0), device=device)[:min(4096, item_emb.size(0))]
    sample = item_emb[ridx]
    sample_n = test_model.normalize(sample)
    encoded_sample = test_model.encoder(sample_n)
    test_model.codebook.kmeans_init(encoded_sample)
print("K-means initialization complete")

# Train WITH quantization but WITHOUT VQ losses
optimizer = torch.optim.Adam(test_model.parameters(), lr=1e-3)

print(f"\nTraining WITH quantization but NO VQ losses (recon only)...\n")

test_model.train()
batch_size = 1024
N = item_emb.size(0)

for epoch in range(1, 51):
    perm = torch.randperm(N, device=device)
    total_recon = 0.0

    for i in range(0, N, batch_size):
        batch = item_emb[perm[i:i+batch_size]]

        # Normalize input
        x_norm = (batch - test_model.x_mean) / (test_model.x_std + 1e-8)

        # Encoder -> Quantize -> Decoder
        z = test_model.encoder(x_norm)
        if test_model.training:
            z = test_model.pre_q_dropout(z)

        # Quantize (but we'll ignore the VQ losses)
        q, codes, commit_loss, codebook_loss = test_model.codebook.forward_with_losses(z)
        x_recon = test_model.decoder(q)

        # ONLY reconstruction loss (ignore commit_loss and codebook_loss)
        loss = F.mse_loss(x_recon, x_norm)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(test_model.parameters(), 1.0)
        optimizer.step()

        total_recon += loss.item() * batch.size(0)

    avg_recon = total_recon / N

    if epoch % 5 == 0 or epoch == 1:
        # Check code diversity
        with torch.no_grad():
            z_all = test_model.encoder(test_model.normalize(item_emb))
            _, codes_all = test_model.codebook(z_all)
            unique = len(torch.unique(codes_all, dim=0))
            diversity = 100 * unique / len(codes_all)

        print(f"Epoch {epoch:2d}/50: MSE = {avg_recon:.6f}, Diversity = {diversity:.1f}% ({unique}/{len(codes_all)})")

print("\n" + "=" * 60)
print("INTERPRETATION:")
print("=" * 60)
if avg_recon < 0.6:
    print("✅ MSE decreased with quantization (no VQ loss)!")
    print("   → VQ losses (commit_loss + codebook_loss) are HURTING reconstruction")
    print("   → Need to reduce beta or fix quantization mechanism")
elif avg_recon < 0.9:
    print("⚠️  MSE improved but quantization adds some degradation")
    print("   → Quantization bottleneck, but VQ losses may also be issue")
else:
    print("❌ MSE stuck even without VQ losses")
    print("   → Quantization itself is breaking gradients")
print("=" * 60)


In [ ]:
# Diagnostic 1: Encoder diversity analysis (FIXED version)
print("=== ENCODER DIVERSITY ANALYSIS ===")
with torch.no_grad():
    sample_size = min(200, len(item_emb))
    enc = model.encoder(model.normalize(item_emb[:sample_size]))
    D = torch.cdist(enc, enc)
    mask = torch.eye(D.size(0), device=D.device).bool()
    mean_offdiag = D[~mask].mean().item()
    nn_dist = (D + torch.eye(D.size(0), device=D.device)*1e6).min(dim=1).values.mean().item()
    print(f"Encoder mean off-diagonal distance: {mean_offdiag:.4f}")
    print(f"Encoder mean nearest-neighbor distance: {nn_dist:.4f}")
    print(f"Range: [{D[~mask].min().item():.4f}, {D[~mask].max().item():.4f}]")

# Diagnostic 2: Per-level usage & perplexity (full dataset)
print("\n=== PER-LEVEL USAGE & PERPLEXITY ===")
with torch.no_grad():
    z = model.encoder(model.normalize(item_emb))
    _, codes = model.codebook(z)  # [N, levels]
    for l in range(codes.size(1)):
        hist = torch.bincount(codes[:, l], minlength=cfg.rqvae_codebook_size).float()
        active = int((hist > 0).sum().item())
        p = (hist / hist.sum().clamp_min(1)).clamp_min(1e-12)
        perplex = torch.exp(-(p * p.log()).sum()).item()
        # Top 3 most used codes
        top3_vals, top3_idx = hist.topk(3)
        print(f"Level {l}: active={active}/{cfg.rqvae_codebook_size}, perplexity={perplex:.1f}")
        print(f"  Top 3 codes: {top3_idx.tolist()} with counts {top3_vals.int().tolist()}")

# Diagnostic 3: Collision profile over triples
print("\n=== COLLISION PROFILE ===")
import numpy as np
import collections
triples = [tuple(c.tolist()) for c in codes.cpu().numpy()]
cnt = collections.Counter(triples)
vals = np.array(list(cnt.values()))
print(f"Unique triples: {len(cnt)}")
print(f"Items per (c1,c2,c3): median={np.median(vals):.1f}, mean={np.mean(vals):.1f}")
print(f"  p90={np.percentile(vals,90):.1f}, p95={np.percentile(vals,95):.1f}, max={vals.max()}")
# Show most collided codes
most_collided = cnt.most_common(5)
print(f"Most collided codes:")
for code, count in most_collided:
    print(f"  {code}: {count} items")

# Diagnostic 4: Neighbor preservation test
print("\n=== NEIGHBOR PRESERVATION TEST ===")
import random
torch.manual_seed(42); random.seed(42)
sample_size = min(1000, len(item_emb))
idx = torch.randperm(item_emb.size(0))[:sample_size]
emb = item_emb[idx]
dist = torch.cdist(emb, emb)
nn_idx = dist.topk(k=6, largest=False).indices[:,1:]  # 5 NNs per item (exclude self)
codes_sub = codes[idx]
share_any = []
for i in range(idx.size(0)):
    base = codes_sub[i]
    for j in nn_idx[i]:
        share_any.append(int((base == codes_sub[j]).any().item()))
lift = np.mean(share_any)
# Random baseline
rand_j = torch.randint(0, idx.size(0), nn_idx.shape, device=nn_idx.device)
share_any_rand = []
for i in range(idx.size(0)):
    base = codes_sub[i]
    for j in rand_j[i]:
        share_any_rand.append(int((base == codes_sub[j]).any().item()))
lift_rand = np.mean(share_any_rand)
print(f"NN share-any-code: {lift:.3f} vs random {lift_rand:.3f}, lift={(lift/(lift_rand+1e-9)):.2f}x")

print("\n=== DIAGNOSTICS COMPLETE ===")

In [ ]:
# CRITICAL DIAGNOSTIC: Verify quantization mechanism is working
print("=" * 60)
print("QUANTIZATION MECHANISM VERIFICATION")
print("=" * 60)

with torch.no_grad():
    # Get a sample of diverse items
    sample_size = 100
    sample_emb = item_emb[:sample_size]

    # Step 1: Check encoder outputs are distinct
    z = model.encoder(model.normalize(sample_emb))
    # No LayerNorm - encoder outputs used directly

    print("\n[1] ENCODER OUTPUT DIVERSITY (latent space)")
    print(f"  Shape: {z.shape}")
    print(f"  Mean: {z.mean(dim=0)[:5].cpu().numpy()}")
    print(f"  Std: {z.std(dim=0)[:5].cpu().numpy()}")
    dist_z = torch.cdist(z[:10], z[:10])
    print(f"  Pairwise distances (first 10): min={dist_z[dist_z>0].min():.4f}, max={dist_z.max():.4f}")

    # Step 2: Check what happens with/without normalization
    print("\n[2] RESIDUAL NORMALIZATION IMPACT (Level 0)")
    r = z.clone()

    # WITHOUT normalization (CURRENT IMPLEMENTATION)
    emb = model.codebook.codebooks[0]
    x2 = (r**2).sum(dim=1, keepdim=True)
    e2 = (emb**2).sum(dim=1)
    scores_no_norm = x2 + e2 - 2 * r @ emb.t()
    idx_no_norm = scores_no_norm.argmin(dim=1)
    print(f"  WITHOUT norm: unique codes = {len(torch.unique(idx_no_norm))}")
    print(f"  Code distribution: {torch.bincount(idx_no_norm, minlength=256)[:10].cpu().numpy()}")

    # WITH normalization (HYPOTHETICAL - not used in actual model)
    r_norm = r / (r.std(dim=0, keepdim=True) + 1e-6)
    x2_norm = (r_norm**2).sum(dim=1, keepdim=True)
    scores_norm = x2_norm + e2 - 2 * r_norm @ emb.t()
    idx_norm = scores_norm.argmin(dim=1)
    print(f"  WITH norm: unique codes = {len(torch.unique(idx_norm))}")
    print(f"  Code distribution: {torch.bincount(idx_norm, minlength=256)[:10].cpu().numpy()}")

    # Step 3: Check codebook diversity
    print("\n[3] CODEBOOK DIVERSITY")
    for l in range(3):
        cb = model.codebook.codebooks[l]
        cb_dist = torch.cdist(cb, cb)
        cb_dist_off_diag = cb_dist[~torch.eye(256, dtype=bool, device=cb.device)]
        print(f"  Level {l}:")
        print(f"    Codebook mean: {cb.mean(dim=0)[:5].cpu().numpy()}")
        print(f"    Codebook std: {cb.std(dim=0)[:5].cpu().numpy()}")
        print(f"    Pairwise distances: min={cb_dist_off_diag.min():.4f}, max={cb_dist_off_diag.max():.4f}, mean={cb_dist_off_diag.mean():.4f}")
        # Check if codebook collapsed to single point
        if cb_dist_off_diag.max() < 1e-3:
            print(f"    ⚠️  CODEBOOK COLLAPSED! All entries are nearly identical")

    # Step 4: Check distance computation details
    print("\n[4] DISTANCE COMPUTATION SANITY CHECK")
    print(f"  Residual shape: {r.shape}")
    print(f"  Codebook shape: {emb.shape}")
    print(f"  Distance matrix shape: {scores_no_norm.shape}")
    print(f"  Min distance per item: {scores_no_norm.min(dim=1).values[:10].cpu().numpy()}")
    print(f"  Assigned codes: {idx_no_norm[:20].cpu().numpy()}")

# Step 5: Check if gradient flow is broken (separate context - needs gradients)
print("\n[5] GRADIENT FLOW CHECK")
model.train()
sample_batch = item_emb[:16].clone().detach().requires_grad_(False)
_, loss, recon, codes_train = model(sample_batch)
print(f"  Training mode - Loss: {loss.item():.6f}, Recon: {recon.item():.6f}")
print(f"  Codes assigned: {codes_train[0].cpu().numpy()}")

# Check gradients
try:
    loss.backward()
    enc_grad_norm = torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), float('inf'))
    cb_grad_norm = torch.nn.utils.clip_grad_norm_(model.codebook.parameters(), float('inf'))
    print(f"  Encoder grad norm: {enc_grad_norm:.6f}")
    print(f"  Codebook grad norm: {cb_grad_norm:.6f}")

    # Check individual parameter gradients
    enc_has_grad = sum(1 for p in model.encoder.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
    cb_has_grad = sum(1 for p in model.codebook.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
    print(f"  Encoder params with gradients: {enc_has_grad}")
    print(f"  Codebook params with gradients: {cb_has_grad}")
except Exception as e:
    print(f"  ⚠️  Gradient computation failed: {e}")
finally:
    model.zero_grad()
    model.eval()

print("=" * 60)

In [ ]:
# Acceptance Criteria Checks
print("=" * 60)
print("ACCEPTANCE CRITERIA EVALUATION")
print("=" * 60)

import numpy as np
import torch

def acceptance_checks(item_emb, codes, cfg):
    N = len(codes)
    levels = codes.size(1)
    checks_passed = []

    # 1) Active codes per level - target: ≥80/256 (31%)
    print("\n[1] ACTIVE CODES PER LEVEL")
    print("-" * 40)
    active_ok = True
    for l in range(levels):
        hist = torch.bincount(codes[:, l], minlength=cfg.rqvae_codebook_size).float()
        active = int((hist > 0).sum().item())
        pct = 100 * active / cfg.rqvae_codebook_size
        status = "✅ PASS" if active >= 80 else "❌ FAIL"
        print(f"  Level {l}: {active}/{cfg.rqvae_codebook_size} ({pct:.1f}%) {status}")
        if active < 80:
            active_ok = False
    checks_passed.append(("Active codes ≥80/level", active_ok))

    # 2) Perplexity targets - target: ≥50 per level
    print("\n[2] PERPLEXITY PER LEVEL")
    print("-" * 40)
    perplex_ok = True
    for l in range(levels):
        hist = torch.bincount(codes[:, l], minlength=cfg.rqvae_codebook_size).float()
        p = (hist / hist.sum().clamp_min(1)).clamp_min(1e-12)
        perplex = torch.exp(-(p * p.log()).sum()).item()
        status = "✅ PASS" if perplex >= 50 else "❌ FAIL"
        print(f"  Level {l}: {perplex:.1f} {status}")
        if perplex < 50:
            perplex_ok = False
    checks_passed.append(("Perplexity ≥50/level", perplex_ok))

    # 3) Collision distribution - target: median ≤2, p90 ≤10
    print("\n[3] COLLISION DISTRIBUTION")
    print("-" * 40)
    from collections import Counter
    triples = [tuple(c.tolist()) for c in codes.cpu().numpy()]
    vals = np.array(list(Counter(triples).values()))
    med, p90, mx = np.median(vals), np.percentile(vals, 90), np.max(vals)
    unique_pct = 100 * len(vals) / N
    print(f"  Unique triples: {len(vals)}/{N} ({unique_pct:.1f}%)")
    print(f"  Items per triple: median={med:.1f}, p90={p90:.1f}, max={mx}")
    coll_ok = (med <= 2 and p90 <= 10)
    status = "✅ PASS" if coll_ok else "❌ FAIL"
    print(f"  Status: {status}")
    checks_passed.append(("Low collision (med≤2, p90≤10)", coll_ok))

    # 4) Overall diversity - target: ≥30% unique codes
    print("\n[4] OVERALL DIVERSITY")
    print("-" * 40)
    diversity_ok = unique_pct >= 30
    status = "✅ PASS" if diversity_ok else "❌ FAIL"
    print(f"  Unique code ratio: {unique_pct:.1f}% {status}")
    checks_passed.append(("Diversity ≥30%", diversity_ok))

    # Summary
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    for check_name, passed in checks_passed:
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"  {check_name}: {status}")

    all_passed = all(p for _, p in checks_passed)
    print("\n" + "=" * 60)
    if all_passed:
        print("🎉 ALL CHECKS PASSED! Model is performing well.")
    else:
        print("⚠️  SOME CHECKS FAILED. Recommendations:")
        if not active_ok or not perplex_ok:
            print("  • Increase training epochs (try 50-100)")
            print("  • Increase beta to 0.5-0.75")
            print("  • Add dead-code revival mechanism")
        if not coll_ok or not diversity_ok:
            print("  • Add LayerNorm before quantization")
            print("  • Use k-means initialization")
            print("  • Increase dropout to 0.2")
    print("=" * 60)

    return all_passed

acceptance_passed = acceptance_checks(item_emb, codes, cfg)

In [ ]:
# Assign Semantic IDs & save maps
import numpy as np
from tiger_semantic_id.src.semantic_id import assign_semantic_ids
sid, sid_to_items, prefix_to_items = assign_semantic_ids(codes, Paths.artifacts_dir, codebook_size=cfg.rqvae_codebook_size)
sid.shape, len(sid_to_items)


In [ ]:
# Visualizations: c1 vs category, and hierarchy
from tiger_semantic_id.src.visualize import plot_c1_category_distribution, plot_hierarchy_c1_c2
fig1 = plot_c1_category_distribution(codes.cpu().numpy(), items)
fig1.savefig(f"{Paths.artifacts_dir}/figs_c1_category.png")
c1_vals = list(pd.Series(codes[:,0].cpu().numpy()).value_counts().head(3).index)
fig2 = plot_hierarchy_c1_c2(codes.cpu().numpy(), items, c1_vals)
fig2.savefig(f"{Paths.artifacts_dir}/figs_hierarchy.png")
fig1, fig2


In [ ]:
# Sequence dataset for generative retrieval
from collections import defaultdict
from tiger_semantic_id.src.seq2seq import TIGERSeqDataset, VocabConfig, Seq2SeqConfig
user_hist = defaultdict(list)
for r in train_df.sort_values(['user_idx','ts']).itertuples(index=False):
    user_hist[int(r.user_idx)].append(int(r.item_idx))
# Fix: use cfg.rqvae_levels to match the RQ-VAE configuration (3 levels)
vocab_cfg = VocabConfig(codebook_size=cfg.rqvae_codebook_size, levels=cfg.rqvae_levels, user_vocab_hash=cfg.user_vocab_hash)
seq_cfg = Seq2SeqConfig(d_model=cfg.seq2seq_d_model, ff=cfg.seq2seq_ff, heads=cfg.seq2seq_heads, layers_enc=cfg.seq2seq_layers_enc, layers_dec=cfg.seq2seq_layers_dec, dropout=cfg.seq2seq_dropout, max_hist_len=cfg.max_hist_len, batch_size=cfg.seq2seq_batch_size, lr=cfg.seq2seq_lr)
train_ds = TIGERSeqDataset(user_hist, sid, user_hash_size=vocab_cfg.user_vocab_hash, codebook_size=vocab_cfg.codebook_size, max_hist_len=seq_cfg.max_hist_len)
len(train_ds), train_ds[0][1][:8], train_ds[0][2]

In [ ]:
# Seq2Seq model & training (compact)
import torch
from torch.utils.data import DataLoader
from tiger_semantic_id.src.seq2seq import TinyTransformer, collate_batch
V = 1 + vocab_cfg.semantic_vocab + vocab_cfg.user_vocab_hash + 2  # PAD=0, BOS=1, then others
model = TinyTransformer(vocab_size=V, d_model=seq_cfg.d_model, ff=seq_cfg.ff, heads=seq_cfg.heads, layers_enc=seq_cfg.layers_enc, layers_dec=seq_cfg.layers_dec, dropout=seq_cfg.dropout)
model = model.cuda() if torch.cuda.is_available() else model
opt = torch.optim.Adam(model.parameters(), lr=seq_cfg.lr)
loader = DataLoader(train_ds, batch_size=seq_cfg.batch_size, shuffle=True, collate_fn=collate_batch)
steps = 0
for src, tgt in loader:
    if torch.cuda.is_available(): src, tgt = src.cuda(), tgt.cuda()
    logits = model(src, tgt[:, :-1])
    loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=0)
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    steps += 1
    if steps % 200 == 0:
        print('steps', steps, 'loss', float(loss))
    if steps >= 1000: break  # knob for Colab runtime
torch.save(model.state_dict(), f"{Paths.artifacts_dir}/seq2seq.pt")
steps


In [ ]:
# Debug: Check token ranges and vocabulary bounds
print("=== TOKEN ANALYSIS ===")
print(f"Vocabulary config:")
print(f"  codebook_size: {vocab_cfg.codebook_size}")
print(f"  levels: {vocab_cfg.levels}")
print(f"  semantic_vocab: {vocab_cfg.semantic_vocab}")
print(f"  user_vocab_hash: {vocab_cfg.user_vocab_hash}")

# Check vocabulary size calculation
V = 1 + vocab_cfg.semantic_vocab + vocab_cfg.user_vocab_hash +2
print(f"  Total vocab size V: {V}")

# Check sample tokens from dataset
sample_data = train_ds[0]
user_tok, seq, tgt = sample_data
print(f"\nSample sequence tokens:")
print(f"  User token: {user_tok}")
print(f"  Sequence: {seq[:10]}...")  # first 10 tokens
print(f"  Target: {tgt}")
print(f"  Max token in sequence: {max(seq) if seq else 'N/A'}")
print(f"  Max token in target: {max(tgt) if tgt else 'N/A'}")

# Check a few more samples
max_tokens = []
for i in range(min(10, len(train_ds))):
    _, seq, tgt = train_ds[i]
    if seq: max_tokens.append(max(seq))
    if tgt: max_tokens.append(max(tgt))

print(f"\nMax tokens across samples: {max(max_tokens) if max_tokens else 'N/A'}")
print(f"Vocab size check: max_token < V? {max(max_tokens) < V if max_tokens else 'N/A'}")

In [ ]:
# Debug: Check RQ-VAE codes
print("=== RQ-VAE CODES ANALYSIS ===")
print(f"Codes shape: {codes.shape}")
print(f"Codes dtype: {codes.dtype}")
print(f"Min codes per level: {codes.min(axis=0)}")
print(f"Max codes per level: {codes.max(axis=0)}")
print(f"Sample codes (first 5 items):")
for i in range(min(5, codes.shape[0])):
    print(f"  Item {i}: {codes[i]}")
print(f"All codes should be in range [0, {cfg.rqvae_codebook_size-1}]")

In [ ]:
# Generate additional artifacts needed for LLM fine-tuning pipeline
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

print("=" * 60)
print("GENERATING LLM PIPELINE ARTIFACTS")
print("=" * 60)

# 1. Create item_to_sid.json (reverse mapping of sid_to_items)
print("\n[1] Creating item_to_sid.json...")
item_to_sid = {}
for i, sid_tuple in enumerate(sid):
    # Convert numpy array to list for JSON serialization
    item_to_sid[str(i)] = sid_tuple.tolist()

item_to_sid_path = Path(Paths.artifacts_dir) / "item_to_sid.json"
with open(item_to_sid_path, "w") as f:
    json.dump(item_to_sid, f)
print(f"✅ Saved {len(item_to_sid)} item->SID mappings to {item_to_sid_path}")
print(f"   Example: item_0 -> {item_to_sid['0']}")

# 2. Create user_sequences.json from train_df
print("\n[2] Creating user_sequences.json...")
user_sequences = defaultdict(list)

# Collect all user sequences from train/val/test (chronologically)
all_interactions = pd.concat([
    train_df[['user_idx', 'item_idx', 'ts']],
    val_df[['user_idx', 'item_idx', 'ts']],
    test_df[['user_idx', 'item_idx', 'ts']]
]).sort_values(['user_idx', 'ts'])

for row in all_interactions.itertuples(index=False):
    user_id = int(row.user_idx)
    item_id = int(row.item_idx)
    user_sequences[user_id].append(item_id)

# Convert to regular dict for JSON serialization
user_sequences = {str(k): v for k, v in user_sequences.items()}

user_seq_path = Path(Paths.artifacts_dir) / "user_sequences.json"
with open(user_seq_path, "w") as f:
    json.dump(user_sequences, f)

total_interactions = sum(len(v) for v in user_sequences.values())
avg_seq_len = total_interactions / len(user_sequences)
print(f"✅ Saved sequences for {len(user_sequences)} users to {user_seq_path}")
print(f"   Total interactions: {total_interactions}")
print(f"   Average sequence length: {avg_seq_len:.1f}")

# 3. Create item_metadata.json (for LLM Types A & B)
print("\n[3] Creating item_metadata.json...")
item_metadata = {}

# Extract metadata from items DataFrame (already merged with meta in earlier cell)
for row in items.itertuples():
    item_idx = int(row.item_idx)

    # Get title (required for Types A & B)
    title = getattr(row, 'title', None)
    if pd.isna(title) or not title:
        continue  # Skip items without titles

    # Store metadata aligned with item_idx
    item_metadata[str(item_idx)] = {
        'title': str(title),
        'category': getattr(row, 'category', []) if hasattr(row, 'category') else [],
        'brand': str(getattr(row, 'brand', '')) if hasattr(row, 'brand') and not pd.isna(getattr(row, 'brand', None)) else '',
        'price': str(getattr(row, 'price', '')) if hasattr(row, 'price') and not pd.isna(getattr(row, 'price', None)) else ''
    }

metadata_path = Path(Paths.artifacts_dir) / "item_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(item_metadata, f)

coverage = 100 * len(item_metadata) / len(items) if len(items) > 0 else 0
print(f"✅ Saved metadata for {len(item_metadata)}/{len(items)} items ({coverage:.1f}% coverage)")
print(f"   Output: {metadata_path}")
if item_metadata:
    sample_key = list(item_metadata.keys())[0]
    sample_title = item_metadata[sample_key]['title'][:80]
    print(f"   Example: item_{sample_key} -> \"{sample_title}...\"")

# 4. Verify all required files exist
print("\n[4] Verifying LLM pipeline prerequisites...")
required_files = [
    "semantic_ids.npy",
    "sid_to_items.json",
    "item_to_sid.json",
    "user_sequences.json",
    "item_metadata.json"
]

all_exist = True
for filename in required_files:
    filepath = Path(Paths.artifacts_dir) / filename
    exists = filepath.exists()
    status = "✅" if exists else "❌"
    print(f"   {status} {filename}")
    if not exists:
        all_exist = False

print("\n" + "=" * 60)
if all_exist:
    print("🎉 ALL LLM PIPELINE PREREQUISITES READY!")
    print(f"\nYou can now run: TIGER_SemanticID_LLM_finetune.ipynb")
    print(f"\nWith metadata coverage of {coverage:.1f}%, you can use:")
    if coverage >= 50:
        print("  - All data types: A,B,C,D,E (~870K examples)")
    else:
        print("  - Fallback types: C,D,E (~680K examples)")
else:
    print("⚠️  SOME FILES MISSING - check errors above")
print("=" * 60)